In [106]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

True

# Util

In [107]:
from pathlib import Path
from typing import List

def get_file_path(file_path: str) -> str:
    """
    파일 경로를 절대 경로로 변환하는 함수
    
    Args:
        file_path: 상대 경로 또는 절대 경로
    """
    # 파일 경로 확인 및 절대 경로로 변환
    file_path = Path(file_path)
    if not file_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        file_path = project_root / file_path
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")
    
    return str(file_path)


def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)

# PDF 파서

In [108]:
# pdfplumber 파서
def extract_text_from_pdf_with_pdfplumber(pdf_path: str, password: str = None) -> str:
    """
    pdfplumber를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    pdfplumber는 PDF 파일을 텍스트 데이터로 추출하는 라이브러리로, 표, 이미지, 레이아웃 등을 잘 보존합니다.
    암호화된 PDF와 암호화되지 않은 PDF 모두 처리할 수 있습니다.
    """
    try:
        import pdfplumber
    except ImportError:
        raise ImportError(
            "PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
            "설치 명령: pip install pdfplumber"
        )
    
    markdown_parts = []
    
    try:
        pdf_path = get_file_path(pdf_path)
        # password가 있으면 암호화된 PDF로 처리, 없으면 암호화되지 않은 PDF로 처리
        pdf_kwargs = {"password": password} if password else {}
        
        with pdfplumber.open(pdf_path, **pdf_kwargs) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        raise

# Excel 파서

In [109]:
# excel loader

import os
from pathlib import Path
from typing import Dict, List

def extract_text_from_excel(excel_path: str) -> Dict[str, str]:
    """
    Excel 파일(.xlsx, .xls)에서 모든 시트의 텍스트를 추출하는 함수
    
    Args:
        excel_path: Excel 파일 경로 (상대 경로 또는 절대 경로)
    
    Returns:
        시트 이름을 키로 하고 추출된 텍스트를 값으로 하는 딕셔너리
    
    Raises:
        FileNotFoundError: Excel 파일을 찾을 수 없을 때
        ImportError: 필요한 Excel 라이브러리가 설치되지 않았을 때
    """
    # 파일 경로 확인 및 절대 경로로 변환    
    excel_path = get_file_path(excel_path)

    print(f"excel_path: {excel_path}")  
    
    # 파일 확장자 확인 (Path 객체로 변환)
    excel_path_obj = Path(excel_path)
    file_ext = excel_path_obj.suffix.lower()
    
    # 여러 Excel 라이브러리 시도 (우선순위 순)
    # 1. pandas + openpyxl/xlrd (가장 편리함)
    try:
        import pandas as pd
        
        # 모든 시트 읽기
        if file_ext == '.xlsx':
            excel_file = pd.ExcelFile(str(excel_path), engine='openpyxl')
        elif file_ext == '.xls':
            excel_file = pd.ExcelFile(str(excel_path), engine='xlrd')
        else:
            # 자동 감지
            excel_file = pd.ExcelFile(str(excel_path))
        
        sheets_text = {}
        for sheet_name in excel_file.sheet_names:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)
            # DataFrame을 텍스트로 변환
            text_parts = []
            # 헤더 포함하여 모든 셀의 값을 문자열로 변환
            for idx, row in df.iterrows():
                row_values = [str(val) if pd.notna(val) else '' for val in row.values]
                text_parts.append(' | '.join(row_values))
            
            sheets_text[sheet_name] = '\n'.join(text_parts)

        print("using pandas")
        return sheets_text
    except ImportError as e:
        if 'pandas' in str(e):
            pass  # pandas가 없으면 다음 방법 시도
        elif 'openpyxl' in str(e) or 'xlrd' in str(e):
            # pandas는 있지만 엔진이 없는 경우
            raise ImportError(
                f"Excel 파일을 읽기 위한 엔진이 필요합니다.\n"
                f".xlsx 파일: pip install openpyxl\n"
                f".xls 파일: pip install xlrd"
            )
        else:
            raise
    
    # 2. openpyxl (xlsx 파일용)
    if file_ext == '.xlsx':
        try:
            from openpyxl import load_workbook
            
            workbook = load_workbook(str(excel_path), data_only=True)
            sheets_text = {}
            
            for sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
                text_parts = []
                
                for row in sheet.iter_rows(values_only=True):
                    row_values = [str(val) if val is not None else '' for val in row]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            print("using openpyxl")
            return sheets_text
        except ImportError:
            pass
    
    # 3. xlrd (xls 파일용)
    if file_ext == '.xls':
        try:
            import xlrd
            
            workbook = xlrd.open_workbook(str(excel_path))
            sheets_text = {}
            
            for sheet_name in workbook.sheet_names():
                sheet = workbook.sheet_by_name(sheet_name)
                text_parts = []
                
                for row_idx in range(sheet.nrows):
                    row_values = [str(sheet.cell_value(row_idx, col_idx)) 
                                 for col_idx in range(sheet.ncols)]
                    text_parts.append(' | '.join(row_values))
                
                sheets_text[sheet_name] = '\n'.join(text_parts)
            
            print("using xlrd")
            return sheets_text
        except ImportError:
            pass
    
    # 모든 라이브러리가 없으면 에러
    raise ImportError(
        "Excel 텍스트 추출을 위한 라이브러리가 설치되지 않았습니다.\n"
        "다음 중 하나를 설치해주세요:\n"
        "  - pandas + openpyxl (권장): pip install pandas openpyxl\n"
        "  - pandas + xlrd (.xls 파일용): pip install pandas xlrd\n"
        "  - openpyxl (.xlsx 파일용): pip install openpyxl\n"
        "  - xlrd (.xls 파일용): pip install xlrd"
    )


def get_all_sheets_text(excel_path: str) -> str:
    """
    Excel 파일의 모든 시트 텍스트를 하나의 문자열로 반환하는 편의 함수
    
    Args:
        excel_path: Excel 파일 경로
    
    Returns:
        모든 시트의 텍스트를 합친 문자열
    """
    sheets_dict = extract_text_from_excel(excel_path)
    
    result_parts = []
    for sheet_name, sheet_text in sheets_dict.items():
        result_parts.append(f"=== 시트: {sheet_name} ===")
        result_parts.append(sheet_text)
        result_parts.append("")  # 빈 줄 추가
    
    return '\n'.join(result_parts)


# document loader

In [110]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None) -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        return extract_text_from_pdf_with_pdfplumber(file_path, password)
    
    elif file_ext in ['.xlsx', '.xls']:
        # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
        return get_all_sheets_text(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )




In [111]:
# text 추출 테스트

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127_20260105_182813.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

# _password = None
# _password = '345678'

# document_text = load_document(_document_file_path, _password)
# print(document_text)

# LLM 모델 생성

In [112]:
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

# vLLM 모델 인스턴스 생성
llm = init_chat_model(
    "openai:",
    temperature=0.0,
    top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)

# LLM chain - 변액일임펀드 설정/해지 지시서 여부 검증 체인

In [113]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# 문서에서 추출한 텍스트가 변액일임펀드 설정/해지 지시서 여부를 판단하는 데이터 모델
class GradeDocument(BaseModel):
    """문서에서 추출한 텍스트가 변액일임펀드 설정/해지 지시서 여부를 판단하기 위한 이진 점수"""

    binary_score: str = Field(description="변액일임펀드 설정/해지 지시서 여부를 'yes' 또는 'no'로 판단합니다.")

# GradeDocument 데이터 모델을 사용하여 구조화된 출력을 생성하는 LLM
structured_llm_grader = llm.with_structured_output(GradeDocument)

# 시스템 프롬프트 정의
system_prompt = """당신은 자산운용사에서 변액일임펀드 설정/해지 업무를 담당하는 오퍼레이터 입니다.
수익자가 보낸 문서가 변액일임펀드 설정/해지 지시서 인지 평가하세요.
문서에는 펀드에 대한 확정분 또는 청구분이 있어야 합니다.
확정분이 있을 경우 이에 대한 설정 금액 또는 해지 금액이 있어야 합니다.
청구분이 있을 경우 이에 대한 설정 금액 또는 해지 금액이 있어야 합니다.
문서의 변액일임펀드 설정/해지 지시서 여부를 나타내기 위해 이진 점수인 'yes' 또는 'no'를 부여하세요."""

# 프롬프트 템플릿 생성
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "수익자가 보낸 문서 내용: \n\n {original_text}")
])

# Retrieval 평가기 초기화
chain_retrieval_grader = grade_prompt | structured_llm_grader

In [114]:
# text 추출 태스트

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127_20260105_182813.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

# _password = None
# _password = '345678'

# document_text = load_document(_document_file_path, _password)
# print(document_text)

In [115]:
# 문서 검증 테스트

# document_text = "test"

# relevance_result = chain_retrieval_grader.invoke({"original_text": document_text})
# print(relevance_result)

# Graph State

In [116]:
# Graph state 클래스

from typing_extensions import TypedDict

class State(TypedDict):
    file_path: str # 변액일임펀드 설정/해지 지시서 파일 경로
    password: str # 지시서 암호
    original_text: str # 지시서 원본 텍스트
    markdown_text: str # 지시서 마크다운 텍스트
    relevance_score: str # 지시서 여부 점수 : yes, no
    analysis_report: str # 지시서 분석 결과
    request_data: str # 지시서 청구분 데이터
    settlement_data: str # 지시서 확정분 데이터
    request_setup_data: str # 지시서 청구분 설정 데이터
    request_redemption_data: str # 지시서 청구분 해지 데이터
    settlement_setup_data: str # 지시서 확정분 설정 데이터
    settlement_redemption_data: str # 지시서 확정분 해지 데이터

# Node - 문서 로드

In [117]:
# 노드 - 문서(pdf, xls, xlsx) 로드
def node_load_document(state: State) -> State:
    file_path = state["file_path"]
    password = state["password"]
    original_text = load_document(file_path, password)
    return {"original_text": original_text}

# Node - 문서 관련성 체크

In [118]:
# 노드 - 문서 관련성 체크 노드
def node_check_relevance(state: State) -> State:
    original_text = state["original_text"]
    relevance_score = chain_retrieval_grader.invoke({"original_text": original_text})
    return {"relevance_score": relevance_score}


# Node - markdown 변환

# Node - Data 분석

# Node - 청구분 수집

# Node - 확정분 수집

# Node - 청구분 설정건 수집

# Node - 청구분 해지건 수집

# Node - 확정분 설정건 수집

# Node - 확정분 해지건 수집

# Node - Fail 처리

# Graph 구성

In [119]:
# Graph 생성

from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(State)

In [120]:
# 지시서 문서 파일 로드 노드 생성
graph_builder.add_node("load_document", node_load_document)

# 문서 관련성 체크 노드 생성
graph_builder.add_node("check_relevance", node_check_relevance)

In [121]:
# graph 엣지 설정

graph_builder.add_edge(START, "load_document")

graph_builder.add_edge("load_document", "check_relevance")

graph_builder.add_edge("check_relevance", END)

In [122]:
# graph 컴파일

graph = graph_builder.compile()

# Graph 실행

In [ ]:
# graph 실행

# text 추출 태스트

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127_20260105_182813.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
_password = '345678'
result = graph.invoke({"file_path": _document_file_path, "password": _password})

print(f"   {result["relevance_score"]}")

excel_path: /Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx
using pandas
   binary_score='yes'
